In [ ]:
# ============================================================
# PROJECT: STYLO – DIGITAL TWIN CLOSET
# MODULE: Computer Vision Clothing Understanding
# GOAL: Convert clothing images into structured digital wardrobe
# MODEL: CLIP (ViT-B/32 Vision Transformer)
# ============================================================

In [ ]:
!pip install torch torchvision torchaudio
!pip install git+https://github.com/openai/CLIP.git
!pip install pillow pandas


  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-0jhmlwnm
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-0jhmlwnm
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=5f0d71d7e11ba3b546d788b9fc05a64ed1c02301319e5cdea631fa67da5b585e
  Stored in directory: /tmp/pip-ephem-wheel-cache-gogrg6hr/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [ ]:
import os

os.makedirs("dataset/images", exist_ok=True)
os.makedirs("dataset/text", exist_ok=True)

print("Folders created ✅")


Folders created ✅


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"dataset/images/{filename}")

print("All images uploaded successfully ✅")


Saving black_dress.png to black_dress.png
All images uploaded successfully ✅


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"dataset/images/{filename}")

print("All images uploaded successfully ✅")

Saving black_pant.png to black_pant.png
All images uploaded successfully ✅


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"dataset/images/{filename}")

print("All images uploaded successfully ✅")

Saving black_skirt.png to black_skirt.png
All images uploaded successfully ✅


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"dataset/images/{filename}")

print("All images uploaded successfully ✅")

Saving blue_pants.png to blue_pants.png
All images uploaded successfully ✅


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"dataset/images/{filename}")

print("All images uploaded successfully ✅")

Saving blue_shirt.png to blue_shirt.png
All images uploaded successfully ✅


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"dataset/images/{filename}")

print("All images uploaded successfully ✅")

Saving green_scarf.png to green_scarf.png
All images uploaded successfully ✅


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"dataset/images/{filename}")

print("All images uploaded successfully ✅")

Saving red_frock.png to red_frock.png
All images uploaded successfully ✅


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"dataset/images/{filename}")

print("All images uploaded successfully ✅")

Saving red_kurta.png to red_kurta.png
All images uploaded successfully ✅


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"dataset/images/{filename}")

print("All images uploaded successfully ✅")

Saving red_shirt.png to red_shirt.png
All images uploaded successfully ✅


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"dataset/images/{filename}")

print("All images uploaded successfully ✅")

Saving yellow_top.png to yellow_top.png
All images uploaded successfully ✅


In [ ]:
image_files = os.listdir("dataset/images")
print("Total images found:", len(image_files))
print(image_files)


Total images found: 11
['black_pant.png', 'red_frock.png', 'black_skirt.png', 'red_shirt.png', 'yellow_top.png', 'green_scarf.png', '.ipynb_checkpoints', 'blue_pants.png', 'red_kurta.png', 'black_dress.png', 'blue_shirt.png']


In [ ]:
fashion_prompts = [
    "a photo of a black dress",
    "a photo of black pants",
    "a photo of a black skirt",
    "a photo of blue pants",
    "a photo of a blue shirt",
    "a photo of a green scarf",
    "a photo of a red frock",
    "a photo of a red kurta",
    "a photo of a red shirt",
    "a photo of a yellow top",
    "formal wear",
    "casual wear",
    "traditional clothing",
    "winter clothing",
    "summer clothing"
]

with open("dataset/text/fashion_prompts.txt", "w") as f:
    for prompt in fashion_prompts:
        f.write(prompt + "\n")

print("Text dataset saved ✅")


Text dataset saved ✅


In [ ]:
import torch
import clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

print("CLIP model loaded on", device)


100%|███████████████████████████████████████| 338M/338M [00:09<00:00, 37.6MiB/s]


CLIP model loaded on cpu


In [ ]:
def load_prompts(path):
    with open(path, "r") as f:
        return f.read().splitlines()

prompts = load_prompts("dataset/text/fashion_prompts.txt")

text_tokens = clip.tokenize(prompts).to(device)

with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)

print("Text embeddings ready ✅")


Text embeddings ready ✅


In [ ]:
import pandas as pd

digital_closet = []

print("Processing images...\n")

for img_name in os.listdir("dataset/images"):
    try:
        img_path = f"dataset/images/{img_name}"
        image = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)

        with torch.no_grad():
            image_features = model.encode_image(image)
            image_features /= image_features.norm(dim=-1, keepdim=True)

        similarity = (image_features @ text_features.T).softmax(dim=-1)
        best_match = prompts[similarity.argmax().item()]

        digital_closet.append({
            "user_id": "user_1",
            "image_name": img_name,
            "predicted_description": best_match,
            "embedding_vector": image_features.cpu().numpy().flatten().tolist()
        })

        print(f"✅ Processed: {img_name}")

    except Exception as e:
        print(f"❌ Error with {img_name}: {e}")

print("\nAll images processed 🎉")


Processing images...

✅ Processed: black_pant.png
✅ Processed: red_frock.png
✅ Processed: black_skirt.png
✅ Processed: red_shirt.png
✅ Processed: yellow_top.png
✅ Processed: green_scarf.png
❌ Error with .ipynb_checkpoints: [Errno 21] Is a directory: 'dataset/images/.ipynb_checkpoints'
✅ Processed: blue_pants.png
✅ Processed: red_kurta.png
✅ Processed: black_dress.png
✅ Processed: blue_shirt.png

All images processed 🎉


In [ ]:
df_closet = pd.DataFrame(digital_closet)

print("Total wardrobe items:", len(df_closet))
df_closet.head(10)


Total wardrobe items: 10


,user_id,image_name,predicted_description,embedding_vector
0,user_1,black_pant.png,a photo of black pants,"[0.02115955762565136, 0.016219256445765495, -0..."
1,user_1,red_frock.png,a photo of a red frock,"[-0.009084107354283333, -0.0056019932962954044..."
2,user_1,black_skirt.png,a photo of a black skirt,"[-0.012076466344296932, 0.011521792970597744, ..."
3,user_1,red_shirt.png,a photo of a red shirt,"[0.003216699929907918, 0.018797507509589195, -..."
4,user_1,yellow_top.png,a photo of a yellow top,"[0.012179323472082615, 0.0577954426407814, -0...."
5,user_1,green_scarf.png,a photo of a green scarf,"[0.01746516488492489, 0.02915756031870842, -0...."
6,user_1,blue_pants.png,a photo of blue pants,"[0.019034426659345627, -0.008146489970386028, ..."
7,user_1,red_kurta.png,a photo of a red kurta,"[-0.0009749227319844067, 0.04431094601750374, ..."
8,user_1,black_dress.png,a photo of a black dress,"[-0.01482441183179617, 0.002452733227983117, 0..."
9,user_1,blue_shirt.png,a photo of a blue shirt,"[0.003993007820099592, 0.02055962383747101, -0..."


In [ ]:
df_closet.to_csv("digital_closet_dataset.csv", index=False)
print("Digital wardrobe saved successfully ✅")


Digital wardrobe saved successfully ✅
